<a href="https://colab.research.google.com/github/mogullamkr21-del/AUTOMATION-EB-DATA/blob/main/DATA_SCRAPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Live Webscraping & Direct Google Sheets Update

This script will authenticate your Google Account, fetch the data from the provided URL, and update the specific sheet **'ASSIGNED-NO MOMENT'**.

In [ ]:
# 1. Install necessary libraries
!pip install gspread pandas requests beautifulsoup4 openpyxl lxml --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 21.1 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.5
    Uninstalling beautifulsoup4-4.13.5:
      Successfully uninstalled beautifulsoup4-4.13.5
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [18]:
import os
import time
import io
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import auth
import gspread
from google.auth import default
from urllib.parse import urljoin

# Authenticate to Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Spreadsheet Details
spreadsheet_url = "https://docs.google.com/spreadsheets/d/1JUGOeS8oAISAXx1nx48JvQa2tE-iBWJnPrcMQ8ICWiM/edit?gid=961131389#gid=961131389"
sheet_name = "UNASSIGNED"

# Credentials provided
username = "MADHUKUMAR"
password = "Madhukumar143"

# Explicit login and list URLs
login_url = 'http://13.234.162.160/AdministratorAccount/AdminLogin.aspx'
target_url = 'http://13.234.162.160/AdministratorAccount/InwardList.aspx'

def run_requests_downloader():
    print("Starting multi-step ASP.NET Session Authenticator...")
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Origin': 'http://13.234.162.160',
        'Referer': login_url
    })

    # Step 1: Request the login page to initialize session tokens
    print("Requesting login landing page...")
    resp = session.get(login_url)
    soup = BeautifulSoup(resp.text, 'html.parser')

    viewstate = soup.find(id='__VIEWSTATE')['value'] if soup.find(id='__VIEWSTATE') else ''
    validation = soup.find(id='__EVENTVALIDATION')['value'] if soup.find(id='__EVENTVALIDATION') else ''
    generator = soup.find(id='__VIEWSTATEGENERATOR')['value'] if soup.find(id='__VIEWSTATEGENERATOR') else ''

    # Dynamically resolve form URL
    form_element = soup.find('form')
    action_relative = form_element.get('action') if form_element else './AdminLogin.aspx'
    post_url = urljoin(login_url, action_relative)
    print(f"Resolved dynamic login POST target: {post_url}")

    payload = {
        '__LASTFOCUS': '',
        '__EVENTTARGET': '',
        '__EVENTARGUMENT': '',
        '__VIEWSTATE': viewstate,
        '__VIEWSTATEGENERATOR': generator,
        '__EVENTVALIDATION': validation,
        'txtUsername': username,
        'txtPassword': password,
        'btnLogin': 'Login'
    }

    # Step 2: Post credentials
    print("Submitting authentication credentials...")
    post_resp = session.post(post_url, data=payload)
    print(f"Post redirection target: {post_resp.url}")

    # Step 3: Access target table page to obtain page viewstate and validation tokens
    print("Fetching inward list table page to prepare export postback...")
    target_resp = session.get(target_url)
    soup_target = BeautifulSoup(target_resp.text, 'html.parser')

    t_viewstate = soup_target.find(id='__VIEWSTATE')['value'] if soup_target.find(id='__VIEWSTATE') else ''
    t_validation = soup_target.find(id='__EVENTVALIDATION')['value'] if soup_target.find(id='__EVENTVALIDATION') else ''
    t_generator = soup_target.find(id='__VIEWSTATEGENERATOR')['value'] if soup_target.find(id='__VIEWSTATEGENERATOR') else ''

    # Step 4: Perform the download Excel postback (triggers the full UNASSIGNED dataset export)
    print("Triggering the Excel export postback for all Unassigned records...")
    export_payload = {
        '__EVENTTARGET': 'ctl00$ContentPlaceHolder1$btnUnAssignExcelDownload',
        '__EVENTARGUMENT': '',
        '__VIEWSTATE': t_viewstate,
        '__VIEWSTATEGENERATOR': t_generator,
        '__EVENTVALIDATION': t_validation
    }

    # Explicit headers to mimic the download button event
    export_headers = {
        'Referer': target_url,
        'Origin': 'http://13.234.162.160',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
    }

    export_resp = session.post(target_url, data=export_payload, headers=export_headers)
    return export_resp.content, export_resp.headers

# Execute scraping and download strategy
try:
    file_bytes, headers = run_requests_downloader()
    content_type = headers.get('Content-Type', '')
    print(f"Response received. Content-Type: {content_type} | Length: {len(file_bytes)} bytes")

    # Decode with fallback encoding check
    content_decoded = ""
    for encoding in ['utf-8', 'utf-16', 'latin-1', 'cp1252']:
        try:
            content_decoded = file_bytes.decode(encoding)
            break
        except UnicodeDecodeError:
            continue

    df = pd.DataFrame()
    if "," in content_decoded and "\n" in content_decoded:
        print("Detected CSV/comma-separated format inside the exported file. Parsing lines directly...")
        lines = content_decoded.strip().split('\n')
        if len(lines) > 1 and not lines[0].startswith("RowNumber") and "UniqueID" in lines[1]:
            print(f"Skipping decorative report title: {lines[0]}")
            content_decoded = "\n".join(lines[1:])
        elif len(lines) > 1 and "unassigned" in lines[0].lower():
            print(f"Skipping decorative report title: {lines[0]}")
            content_decoded = "\n".join(lines[1:])

        df = pd.read_csv(io.StringIO(content_decoded), on_bad_lines='skip')
    else:
        try:
            print("Attempting standard binary Excel parsing...")
            df = pd.read_excel(io.BytesIO(file_bytes))
        except Exception as e:
            print(f"Binary parsing failed: {e}. Trying fallback HTML load...")
            tables = pd.read_html(io.BytesIO(file_bytes))
            df = max(tables, key=len) if tables else pd.DataFrame()

    if not df.empty:
        df = df.fillna('')
        df = df.astype(str)

        print("\n--- EXPORTED EXCEL STRUCTURE PREVIEW ---")
        print(df.head(10))

        # Write results directly to Google Sheets
        sh = gc.open_by_url(spreadsheet_url)
        try:
            worksheet = sh.worksheet(sheet_name)
        except gspread.exceptions.WorksheetNotFound:
            print(f"Sheet '{sheet_name}' not found. Creating it...")
            worksheet = sh.add_worksheet(title=sheet_name, rows="100", cols="20")

        worksheet.clear()
        data_to_write = [df.columns.values.tolist()] + df.values.tolist()
        worksheet.update(values=data_to_write, range_name="A1")
        print(f"\nSuccessfully updated Google Sheet '{sheet_name}' with the complete downloaded export structure!")
    else:
        print("No table data could be parsed from the exported file content.")

except Exception as err:
    print(f"Download execution or parsing failed: {err}")

Starting multi-step ASP.NET Session Authenticator...
Requesting login landing page...
Resolved dynamic login POST target: http://13.234.162.160/AdministratorAccount/AdminLogin.aspx?bid=6QOqoV6reu4JL6++H5Q3aw%3d%3d
Submitting authentication credentials...
Post redirection target: http://13.234.162.160/AdministratorAccount/addinwardform.aspx
Fetching inward list table page to prepare export postback...
Triggering the Excel export postback for all Unassigned records...
Response received. Content-Type: Text/vnd.ms-excel; charset=utf-8 | Length: 1960980 bytes
Detected CSV/comma-separated format inside the exported file. Parsing lines directly...
Skipping decorative report title: Electronics Bazaar FZC Unassign Device Report

--- EXPORTED EXCEL STRUCTURE PREVIEW ---
  RowNumber UniqueID BrandName  IMEINumber              ItemName RackNumber  \
0         1  31222.0      DELL     CWQ3P93         LATITUDE 3310              
1         2  31217.0      DELL     J798733         LATITUDE 3410       

### 1. Automation Script (`sync_data.py`)
Save the following python code into a file named `sync_data.py` in your GitHub repository root. This script uses a service account key provided via environment variables to bypass the interactive user authentication required inside Google Colab.

In [19]:
#### Contents of sync_data.py ####
# import os
# import json
# import io
# import requests
# import pandas as pd
# from bs4 import BeautifulSoup
# import gspread
# from google.oauth2.service_account import Credentials
# from urllib.parse import urljoin
#
# # Load service account credentials from environment variables
# creds_json = os.environ.get('GOOGLE_CREDS_JSON')
# if not creds_json:
#     raise ValueError("GOOGLE_CREDS_JSON environment variable is missing!")
#
# info = json.loads(creds_json)
# scopes = ['https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/drive']
# creds = Credentials.from_service_account_info(info, scopes=scopes)
# gc = gspread.authorize(creds)
#
# spreadsheet_url = "https://docs.google.com/spreadsheets/d/1JUGOeS8oAISAXx1nx48JvQa2tE-iBWJnPrcMQ8ICWiM/edit?gid=961131389#gid=961131389"
# sheet_name = "UNASSIGNED"
#
# username = "MADHUKUMAR"
# password = "Madhukumar143"
#
# login_url = 'http://13.234.162.160/AdministratorAccount/AdminLogin.aspx'
# target_url = 'http://13.234.162.160/AdministratorAccount/InwardList.aspx'
#
# def run_requests_downloader():
#     session = requests.Session()
#     session.headers.update({
#         'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
#     })
#     resp = session.get(login_url)
#     soup = BeautifulSoup(resp.text, 'html.parser')
#
#     viewstate = soup.find(id='__VIEWSTATE')['value'] if soup.find(id='__VIEWSTATE') else ''
#     validation = soup.find(id='__EVENTVALIDATION')['value'] if soup.find(id='__EVENTVALIDATION') else ''
#     generator = soup.find(id='__VIEWSTATEGENERATOR')['value'] if soup.find(id='__VIEWSTATEGENERATOR') else ''
#     action_relative = soup.find('form').get('action') if soup.find('form') else './AdminLogin.aspx'
#     post_url = urljoin(login_url, action_relative)
#
#     payload = {
#         '__LASTFOCUS': '',
#         '__EVENTTARGET': '',
#         '__EVENTARGUMENT': '',
#         '__VIEWSTATE': viewstate,
#         '__VIEWSTATEGENERATOR': generator,
#         '__EVENTVALIDATION': validation,
#         'txtUsername': username,
#         'txtPassword': password,
#         'btnLogin': 'Login'
#     }
#
#     session.post(post_url, data=payload)
#     target_resp = session.get(target_url)
#     soup_target = BeautifulSoup(target_resp.text, 'html.parser')
#
#     t_viewstate = soup_target.find(id='__VIEWSTATE')['value'] if soup_target.find(id='__VIEWSTATE') else ''
#     t_validation = soup_target.find(id='__EVENTVALIDATION')['value'] if soup_target.find(id='__EVENTVALIDATION') else ''
#     t_generator = soup_target.find(id='__VIEWSTATEGENERATOR')['value'] if soup_target.find(id='__VIEWSTATEGENERATOR') else ''
#
#     export_payload = {
#         '__EVENTTARGET': 'ctl00$ContentPlaceHolder1$btnUnAssignExcelDownload',
#         '__EVENTARGUMENT': '',
#         '__VIEWSTATE': t_viewstate,
#         '__VIEWSTATEGENERATOR': t_generator,
#         '__EVENTVALIDATION': t_validation
#     }
#
#     export_headers = {
#         'Referer': target_url,
#         'Origin': 'http://13.234.162.160'
#     }
#
#     export_resp = session.post(target_url, data=export_payload, headers=export_headers)
#     return export_resp.content, export_resp.headers
#
# file_bytes, headers = run_requests_downloader()
# content_decoded = ""
# for encoding in ['utf-8', 'utf-16', 'latin-1', 'cp1252']:
#     try:
#         content_decoded = file_bytes.decode(encoding)
#         break
#     except UnicodeDecodeError:
#         continue
#
# df = pd.DataFrame()
# if "," in content_decoded and "\n" in content_decoded:
#     lines = content_decoded.strip().split('\n')
#     if len(lines) > 1 and "unassigned" in lines[0].lower():
#         content_decoded = "\n".join(lines[1:])
#     df = pd.read_csv(io.StringIO(content_decoded), on_bad_lines='skip')
#
# if not df.empty:
#     df = df.fillna('').astype(str)
#     sh = gc.open_by_url(spreadsheet_url)
#     try:
#         worksheet = sh.worksheet(sheet_name)
#     except gspread.exceptions.WorksheetNotFound:
#         worksheet = sh.add_worksheet(title=sheet_name, rows="100", cols="20")
#     worksheet.clear()
#     data_to_write = [df.columns.values.tolist()] + df.values.tolist()
#     worksheet.update(values=data_to_write, range_name="A1")
#     print("Sync successfully finished!")


### 2. GitHub Action YAML file (`.github/workflows/sync.yml`)

Place the file below under `.github/workflows/sync.yml` to trigger the sync every 5 minutes:

```yaml
name: Real-Time Unassigned Sync

on:
  schedule:
    - cron: '*/5 * * * *'
  workflow_dispatch:

jobs:
  run-sync:
    runs-on: ubuntu-latest
    steps:
    - name: Check out repository
      uses: actions/checkout@v3
    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'
    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install gspread pandas requests beautifulsoup4 openpyxl lxml google-auth
    - name: Run Sync Script
      env:
        GOOGLE_CREDS_JSON: ${{ secrets.GOOGLE_CREDS_JSON }}
      run: python sync_data.py
```

In [17]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

login_url = 'http://13.234.162.160/AdministratorAccount/AdminLogin.aspx'
target_url = 'http://13.234.162.160/AdministratorAccount/InwardList.aspx'

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# 1. Login
resp = session.get(login_url)
soup = BeautifulSoup(resp.text, 'html.parser')
viewstate = soup.find(id='__VIEWSTATE')['value'] if soup.find(id='__VIEWSTATE') else ''
validation = soup.find(id='__EVENTVALIDATION')['value'] if soup.find(id='__EVENTVALIDATION') else ''
generator = soup.find(id='__VIEWSTATEGENERATOR')['value'] if soup.find(id='__VIEWSTATEGENERATOR') else ''
action_relative = soup.find('form').get('action') if soup.find('form') else './AdminLogin.aspx'
post_url = urljoin(login_url, action_relative)

payload = {
    '__VIEWSTATE': viewstate,
    '__VIEWSTATEGENERATOR': generator,
    '__EVENTVALIDATION': validation,
    'txtUsername': 'MADHUKUMAR',
    'txtPassword': 'Madhukumar143',
    'btnLogin': 'Login'
}
session.post(post_url, data=payload)

# 2. Access target Inward List page
target_resp = session.get(target_url)
soup_target = BeautifulSoup(target_resp.text, 'html.parser')

# Print all button and link attributes containing 'Excel' or 'Download' or 'Unassigned' to identify the exact __EVENTTARGET
print("--- SEARCHING FOR UNASSIGNED OR EXCEL BUTTON TARGETS ---")
for element in soup_target.find_all(['input', 'a', 'button']):
    elem_str = str(element).lower()
    if any(keyword in elem_str for keyword in ['excel', 'download', 'unassign', 'export']):
        print(f"ID: {element.get('id')} | Name: {element.get('name')} | Value/Text: {element.get('value') or element.text.strip()} | Element: {str(element)[:150]}")


--- SEARCHING FOR UNASSIGNED OR EXCEL BUTTON TARGETS ---
ID: None | Name: None | Value/Text: SKU Bulk Excel Upload | Element: <a href="SKUBulkManagement.aspx" main-menuname="master" menu-parent="true" page-name="skubulkmanagement.aspx"><i class="fa fa-file-excel-o"></i>SKU Bu
ID: None | Name: None | Value/Text: SKU MRP Update | Element: <a href="SKUMRPExcelUpdate.aspx" main-menuname="master" menu-parent="true" page-name="skumrpexcelupdate.aspx"><i class="fa fa-inr"></i>SKU MRP Update<
ID: None | Name: None | Value/Text: Bulk Part Upload | Element: <a href="PartExcelUpload.aspx" main-menuname="part" page-name="partexcelupload.aspx"><i class="fa fa-upload"></i>Bulk Part Upload</a>
ID: None | Name: None | Value/Text: Bulk Device Upload | Element: <a href="InwardExcelUpload.aspx" main-menuname="inward" page-name="inwardexcelupload.aspx"><i class="fa fa-upload"></i>Bulk Device Upload</a>
ID: None | Name: None | Value/Text: Master Inward | Element: <a href="InwardExcelDump.aspx" main-menunam

### GitHub Actions Configuration File (`.github/workflows/daily_sync.yml`)

Create this file in your repository to schedule the execution at a specific UTC time daily:

```yaml
name: Daily Inward List Sync

on:
  schedule:
    # Runs daily at 00:00 UTC (Adjust as needed)
    - cron: '0 0 * * *'
  workflow_dispatch: # Allows you to manually trigger the run anytime via GitHub UI

jobs:
  run-sync:
    runs-on: ubuntu-latest

    steps:
    - name: Check out repository
      uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'

    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install gspread pandas requests beautifulsoup4 openpyxl lxml google-auth

    - name: Run Sync Script
      env:
        # Passes your service account credentials securely
        GOOGLE_APPLICATION_CREDENTIALS_JSON: ${{ secrets.GOOGLE_CREDS_JSON }}
      run: python sync_data.py
```

### Updated GitHub Actions Workflow (`.github/workflows/daily_sync.yml`)

This configuration uses `cron: '*/5 * * * *'` to trigger the workflow every 5 minutes (the maximum frequency allowed by GitHub).

```yaml
name: Real-Time Inward List Sync

on:
  schedule:
    # Runs every 5 minutes (GitHub's minimum scheduler limit)
    - cron: '*/5 * * * *'
  workflow_dispatch: # Allows you to manually trigger the run anytime via GitHub UI

jobs:
  run-sync:
    runs-on: ubuntu-latest

    steps:
    - name: Check out repository
      uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'

    - name: Install Dependencies
      run: |
        python -m pip install --upgrade pip
        pip install gspread pandas requests beautifulsoup4 openpyxl lxml google-auth

    - name: Run Sync Script
      env:
        # Passes your service account credentials securely
        GOOGLE_APPLICATION_CREDENTIALS_JSON: ${{ secrets.GOOGLE_CREDS_JSON }}
      run: python sync_data.py
```

In [13]:


from PIL import Image
import matplotlib.pyplot as plt

# Open and inspect the provided screenshot to locate the download button and format hints
image_path = '/content/Screenshot 2026-11-18 151023.png'
try:
    img = Image.open(image_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Could not load the screenshot: {e}")

Could not load the screenshot: [Errno 2] No such file or directory: '/content/Screenshot 2026-11-18 151023.png'


In [14]:
import requests
from bs4 import BeautifulSoup

login_url = 'http://13.234.162.160/AdministratorAccount/AdminLogin.aspx'
resp = requests.get(login_url)
soup = BeautifulSoup(resp.text, 'html.parser')

print("--- FORM ACTION ---")
form = soup.find('form')
if form:
    print(f"Action: {form.get('action')}")
    print(f"Method: {form.get('method')}")
else:
    print("No form found.")

print("\n--- INPUT FIELDS FOUND ---")
for inp in soup.find_all(['input', 'button']):
    print(f"Tag: {inp.name} | Name: {inp.get('name')} | ID: {inp.get('id')} | Type: {inp.get('type')} | Value: {inp.get('value')[:50] if inp.get('value') else 'None'}")

--- FORM ACTION ---
Action: ./AdminLogin.aspx?bid=6QOqoV6reu4JL6++H5Q3aw%3d%3d
Method: post

--- INPUT FIELDS FOUND ---
Tag: input | Name: __VIEWSTATE | ID: __VIEWSTATE | Type: hidden | Value: iPo/mHxL6EkMMK3yAQ/N7cNit2FBlvHnpnJrWnqLj3l5ixWKSB
Tag: input | Name: __VIEWSTATEGENERATOR | ID: __VIEWSTATEGENERATOR | Type: hidden | Value: DB6ADC15
Tag: input | Name: __EVENTVALIDATION | ID: __EVENTVALIDATION | Type: hidden | Value: yHogl3mMyDl5V8Gcz67DTMDH08xp+3XoJe1L84V6EQnRezz+im
Tag: input | Name: txtUsername | ID: txtUsername | Type: text | Value: None
Tag: input | Name: txtPassword | ID: txtPassword | Type: password | Value: None
Tag: input | Name: btnLogin | ID: btnLogin | Type: submit | Value: Login


In [15]:
# Log back in and fetch the HTML of InwardList.aspx to locate the download button structure and pagination elements
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import io

login_url = 'http://13.234.162.160/AdministratorAccount/AdminLogin.aspx'
target_url = 'http://13.234.162.160/AdministratorAccount/InwardList.aspx'

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# 1. Login
resp = session.get(login_url)
soup = BeautifulSoup(resp.text, 'html.parser')
viewstate = soup.find(id='__VIEWSTATE')['value'] if soup.find(id='__VIEWSTATE') else ''
validation = soup.find(id='__EVENTVALIDATION')['value'] if soup.find(id='__EVENTVALIDATION') else ''
generator = soup.find(id='__VIEWSTATEGENERATOR')['value'] if soup.find(id='__VIEWSTATEGENERATOR') else ''
action_relative = soup.find('form').get('action') if soup.find('form') else './AdminLogin.aspx'
post_url = urljoin(login_url, action_relative)

payload = {
    '__VIEWSTATE': viewstate,
    '__VIEWSTATEGENERATOR': generator,
    '__EVENTVALIDATION': validation,
    'txtUsername': 'MADHUKUMAR',
    'txtPassword': 'Madhukumar',
    'btnLogin': 'Login'
}
session.post(post_url, data=payload)

# 2. Get target page
target_resp = session.get(target_url)
soup_target = BeautifulSoup(target_resp.text, 'html.parser')

# Look for button controls with icons or export keywords
print("--- POTENTIAL EXPORT / DOWNLOAD BUTTONS ---")
for btn in soup_target.find_all(['a', 'button', 'input']):
    btn_str = str(btn)[:200]
    if any(keyword in btn_str.lower() for keyword in ['download', 'export', 'btn', 'fa-download', 'fa-file', 'excel', 'csv']):
        print(f"ID: {btn.get('id')} | Class: {btn.get('class')} | OnClick: {btn.get('onclick')} | HTML: {btn_str}\n")

# Look for pagination elements to scrape all pages
print("--- PAGINATION DETAILS ---")
pager = soup_target.find(class_='pagination') or soup_target.find(id='pagination')
if pager:
    print("Found standard pagination block:")
    print(pager.prettify()[:1000])
else:
    print("No standard pagination class found. Checking table footer / rows representing page numbers...")
    # Check for links inside tables containing javascript:__doPostBack or specific numbers
    page_links = [a for a in soup_target.find_all('a') if 'doPostBack' in str(a.get('href')) or a.text.strip().isdigit()]
    for pl in page_links[:15]:
        print(f"Text: {pl.text.strip()} | Href: {pl.get('href')}")

--- POTENTIAL EXPORT / DOWNLOAD BUTTONS ---
--- PAGINATION DETAILS ---
No standard pagination class found. Checking table footer / rows representing page numbers...


In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

login_url = 'http://13.234.162.160/AdministratorAccount/AdminLogin.aspx'

username = "MADHUKUMAR"
password = "Madhukumar143"

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# 1. Fetch tokens
resp = session.get(login_url)
soup = BeautifulSoup(resp.text, 'html.parser')

viewstate = soup.find(id='__VIEWSTATE')['value'] if soup.find(id='__VIEWSTATE') else ''
validation = soup.find(id='__EVENTVALIDATION')['value'] if soup.find(id='__EVENTVALIDATION') else ''
generator = soup.find(id='__VIEWSTATEGENERATOR')['value'] if soup.find(id='__VIEWSTATEGENERATOR') else ''
action_relative = soup.find('form').get('action') if soup.find('form') else './AdminLogin.aspx'
post_url = urljoin(login_url, action_relative)

payload = {
    '__LASTFOCUS': '',
    '__EVENTTARGET': '',
    '__EVENTARGUMENT': '',
    '__VIEWSTATE': viewstate,
    '__VIEWSTATEGENERATOR': generator,
    '__EVENTVALIDATION': validation,
    'txtUsername': username,
    'txtPassword': password,
    'btnLogin': 'Login'
}

# 2. POST credentials
post_resp = session.post(post_url, data=payload)

# 3. Check for rendered login page and print any error text
soup_result = BeautifulSoup(post_resp.text, 'html.parser')
print(f"Final Response URL: {post_resp.url}")

# Find labels or alert boxes containing error messages
alerts = soup_result.find_all(class_=['alert', 'text-danger', 'error'])
labels = soup_result.find_all('span')

print("\n--- SCANNING FOR LOGIN ERROR MESSAGES ---")
for tag in alerts + labels:
    text = tag.text.strip()
    if text and any(kw in text.lower() for kw in ['invalid', 'user', 'password', 'error', 'fail', 'incorrect']):
        print(f"Found message: '{text}'")


Final Response URL: http://13.234.162.160/AdministratorAccount/addinwardform.aspx

--- SCANNING FOR LOGIN ERROR MESSAGES ---
Found message: 'Change Password'


2 PROJECT FROM SITE UNASSIGNED


Authenticating Google account...
Attempting to discover and pull from data API endpoints...
NEXT_DATA not found, parsing React Server Component stream chunk patterns...
Checking API path: https://electronicsbazaar.vercel.app/api/devices
Checking API path: https://electronicsbazaar.vercel.app/api/data
Checking API path: https://electronicsbazaar.vercel.app/api/inward
Checking API path: https://electronicsbazaar.vercel.app/api/production
Successfully collected 3 rows from React application state!
Sync failed: dup tb
